In [ ]:
import os, sys

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

HOME = "/home/j-j14d208"
CODE_DIR = f"{HOME}/2_ml_pipeline"
DATA_DIR = f"{HOME}/kospi200-project"
os.environ["BASE_PATH"] = DATA_DIR
os.chdir(CODE_DIR)
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

import logging
import utils.logger as _log_mod
def setup_logger(name):
    logger = logging.getLogger(name)
    if logger.handlers: return logger
    logger.setLevel(logging.DEBUG)
    h = logging.StreamHandler(sys.stdout)
    h.setLevel(logging.INFO)
    h.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(name)s | %(message)s", "%H:%M:%S"))
    logger.addHandler(h)
    return logger
_log_mod.setup_logger = setup_logger

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"CWD: {os.getcwd()}")

In [ ]:
!pip install lightning lightgbm --quiet

In [ ]:
"""앙상블 백테스트 v7b — BUY_THRESHOLD 그리드 서치
v7(v2 슬라이딩 메타모델)에 최적 BUY_THRESHOLD 탐색.
Step 1~4는 한 번만 실행, Step 5를 여러 threshold로 반복.
"""
import warnings, json, joblib
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass, field
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from copy import deepcopy

try:
    import lightning.pytorch as pl
except ImportError:
    import pytorch_lightning as pl

warnings.filterwarnings("ignore")
print(f"PyTorch: {torch.__version__}")

In [ ]:
# ===== 설정 =====
BASE_PATH = Path(os.environ.get("BASE_PATH"))
TFT_FEATURE_PATH = BASE_PATH / "processed" / "tft_features" / "tft_features.parquet"
RAW_OHLCV_PATH = BASE_PATH / "raw" / "ohlcv"
RAW_MACRO_PATH = BASE_PATH / "raw" / "macro"
MODEL_SAVE_PATH = BASE_PATH / "models" / "ensemble_backtest"
MODEL_SAVE_PATH.mkdir(parents=True, exist_ok=True)

TFT_CKPT_DIR = BASE_PATH / "models" / "tft_v2_wf" / "checkpoints"
LGBM_CKPT_DIR = BASE_PATH / "models" / "lgbm_wf" / "checkpoints"
GARCH_CKPT_DIR = BASE_PATH / "models" / "garch_wf" / "checkpoints"

BT_START_WINDOW = 1
MAX_ENCODER_LENGTH = 30
BATCH_SIZE = 256

CLEANED_FEATURES = ["vix_change", "vix", "macd_norm", "momentum_20d",
                    "relative_return", "high_low_range", "kospi200_return", "volume_ratio_5d"]

# ===== 고정 파라미터 (v6과 동일) =====
INITIAL_CAPITAL = 100_000_000
BUY_COMMISSION = 0.00015
SELL_COMMISSION = 0.00215
STOP_LOSS = -0.03
TAKE_PROFIT = 0.07
MODEL_EXIT = 0.45
REBUY_COOLDOWN = 5
BASE_HOLD_DAYS = 5
HOLD_EXTEND_THRESHOLD = 0.52
MAX_HOLD_DAYS = 15
EXTENDED_STOP_LOSS = -0.02
EXTENDED_TAKE_PROFIT = 0.10

# v2 슬라이딩 메타모델
SLIDING_WINDOW = 4
META_FEATURES = ["tft_prob", "lgbm_prob", "disagreement", "consensus_strength", "vol_5d", "risk_flag"]

# ===== 그리드 서치 대상 =====
GRID = [
    {"buy_threshold": 0.50, "top_n": 10, "max_positions": 20},
    {"buy_threshold": 0.52, "top_n": 8,  "max_positions": 18},
    {"buy_threshold": 0.54, "top_n": 7,  "max_positions": 15},  # v7 원본
    {"buy_threshold": 0.56, "top_n": 7,  "max_positions": 15},
    {"buy_threshold": 0.58, "top_n": 7,  "max_positions": 15},
    {"buy_threshold": 0.60, "top_n": 5,  "max_positions": 12},
    {"buy_threshold": 0.62, "top_n": 5,  "max_positions": 10},
]
print(f"그리드 서치: {len(GRID)}개 조합")
for g in GRID:
    print(f"  threshold={g['buy_threshold']}, top_n={g['top_n']}, max_pos={g['max_positions']}")

# Walk-Forward 윈도우
WF_START = "2021-01-01"
WF_END = "2026-01-31"
WF_STEP_MONTHS = 3
windows = []
current = pd.Timestamp(WF_START)
end = pd.Timestamp(WF_END)
while current < end:
    test_end = current + pd.DateOffset(months=WF_STEP_MONTHS) - pd.Timedelta(days=1)
    if test_end > end: test_end = end
    train_end = current - pd.Timedelta(days=1)
    windows.append((str(train_end.date()), str(current.date()), str(test_end.date())))
    current += pd.DateOffset(months=WF_STEP_MONTHS)

In [ ]:
# ===== 데이터 로드 =====
df = pd.read_parquet(str(TFT_FEATURE_PATH))
df["date"] = pd.to_datetime(df["date"])
df["target_5d"] = df["target_5d"].astype(int)
CLEANED_FEATURES = [c for c in CLEANED_FEATURES if c in df.columns]
N_FEATURES = len(CLEANED_FEATURES)
print(f"데이터: {len(df):,} rows, {N_FEATURES} features")

In [ ]:
# ===== TFT v2 모델 정의 (로드용) =====
class GatedLinearUnit(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.fc=nn.Linear(d,d); self.gate=nn.Linear(d,d)
    def forward(self, x): return self.fc(x)*torch.sigmoid(self.gate(x))

class GRN(nn.Module):
    def __init__(self, d_in, d_h, d_out, drop=0.1):
        super().__init__()
        self.fc1=nn.Linear(d_in,d_h); self.fc2=nn.Linear(d_h,d_out)
        self.gate=GatedLinearUnit(d_out); self.norm=nn.LayerNorm(d_out)
        self.drop=nn.Dropout(drop); self.skip=nn.Linear(d_in,d_out) if d_in!=d_out else nn.Identity()
    def forward(self, x):
        r=self.skip(x); h=self.drop(F.elu(self.fc2(F.elu(self.fc1(x)))))
        return self.norm(self.gate(h)+r)

class VSN(nn.Module):
    def __init__(self, n_v, d, drop=0.1):
        super().__init__()
        self.grns=nn.ModuleList([GRN(d,d,d,drop) for _ in range(n_v)])
        self.sg=GRN(n_v*d,d,n_v,drop)
    def forward(self, x):
        B,S,V,D=x.shape
        p=torch.stack([self.grns[i](x[:,:,i,:]) for i in range(V)],dim=2)
        w=torch.softmax(self.sg(x.reshape(B,S,V*D)),dim=-1).unsqueeze(-1)
        return (p*w).sum(dim=2)

class TFTv2(pl.LightningModule):
    def __init__(self, n_feat, seq_len=30, d=128, heads=4, n_lstm=1, drop=0.2, n_cls=2, lr=5e-4, cw=None):
        super().__init__()
        self.save_hyperparameters(ignore=["cw"]); self.lr=lr
        self.fe=nn.Linear(1,d); self.vsn=VSN(n_feat,d,drop)
        self.lstm=nn.LSTM(d,d,n_lstm,batch_first=True)
        self.attn=nn.MultiheadAttention(d,heads,dropout=drop,batch_first=True)
        self.an=nn.LayerNorm(d); self.ag=GatedLinearUnit(d)
        self.go=GRN(d,d,d,drop); self.head=nn.Linear(d,n_cls)
        self.loss_fn=nn.CrossEntropyLoss(weight=cw) if cw is not None else nn.CrossEntropyLoss()
    def forward(self, x):
        B,S,F=x.shape; x=self.fe(x.unsqueeze(-1)); x=self.vsn(x)
        x,_=self.lstm(x); a,_=self.attn(x,x,x); x=self.an(x+self.ag(a))
        return self.head(self.go(x[:,-1,:]))
    def training_step(self,b,_): return self.loss_fn(self(b[0]),b[1])
    def configure_optimizers(self): return torch.optim.AdamW(self.parameters(),lr=self.lr)

print("TFT v2 정의 완료")

In [ ]:
# ===== 유틸 함수 =====
def make_tft_samples(full_df, start, end, seq_len, feats):
    samples, metas = [], []
    s=pd.Timestamp(start); e=pd.Timestamp(end) if end else full_df["date"].max()
    for _,g in full_df.groupby("ticker"):
        g=g.sort_values("time_idx"); v=g[feats].values.astype(np.float32)
        t=g["target_5d"].values.astype(np.int64); d=g["date"].values; tk=g["ticker"].values
        for i in range(seq_len,len(g)):
            if d[i]>=s and d[i]<=e:
                samples.append((v[i-seq_len:i],t[i]))
                metas.append({"date": str(d[i])[:10], "ticker": str(tk[i])})
    return samples, metas

class SeqDS(Dataset):
    def __init__(self,s): self.s=s
    def __len__(self): return len(self.s)
    def __getitem__(self,i): x,y=self.s[i]; return torch.tensor(x),torch.tensor(y)

def predict_tft(model, samples):
    loader = DataLoader(SeqDS(samples), batch_size=BATCH_SIZE*2, shuffle=False, num_workers=0)
    ps, ls = [], []
    model.eval(); model.cuda()
    with torch.no_grad():
        for x,y in loader:
            ps.extend(torch.softmax(model(x.cuda()),dim=-1)[:,1].cpu().numpy()); ls.extend(y.numpy())
    return np.array(ps), np.array(ls)

def predict_lgbm(model, full_df, start, end, feats):
    s=pd.Timestamp(start); e=pd.Timestamp(end)
    mask = (full_df["date"]>=s) & (full_df["date"]<=e)
    sub = full_df[mask]
    X = sub[feats].values.astype(np.float32)
    y = sub["target_5d"].values
    probs = model.predict_proba(X)[:, 1]
    metas = [{"date": str(d)[:10], "ticker": t} for d, t in zip(sub["date"].values, sub["ticker"].values)]
    return probs, y, metas

print("유틸 정의 완료")

In [ ]:
# ===== Step 1: 모든 윈도우의 예측 수집 (v7: 파생 피처 추가) =====
all_predictions = []

for i, (train_end, test_start, test_end) in enumerate(windows):
    tft_ckpt = TFT_CKPT_DIR / f"window_{i:02d}_te_{train_end}.ckpt"
    lgbm_ckpt = LGBM_CKPT_DIR / f"window_{i:02d}_te_{train_end}.joblib"
    garch_ckpt = GARCH_CKPT_DIR / f"window_{i:02d}_te_{train_end}.parquet"
    
    if not tft_ckpt.exists() or not lgbm_ckpt.exists():
        print(f"  [{i:2d}] SKIP: 체크포인트 없음"); continue
    
    print(f"  [{i:2d}] {test_start}~{test_end} 예측 생성...")
    
    tft_model = TFTv2.load_from_checkpoint(str(tft_ckpt), strict=False)
    tft_samples, tft_metas = make_tft_samples(df, test_start, test_end, MAX_ENCODER_LENGTH, CLEANED_FEATURES)
    if len(tft_samples) < 10:
        del tft_model; torch.cuda.empty_cache(); continue
    tft_probs, tft_labels = predict_tft(tft_model, tft_samples)
    del tft_model; torch.cuda.empty_cache()
    
    lgbm_model = joblib.load(str(lgbm_ckpt))
    lgbm_probs, _, lgbm_metas = predict_lgbm(lgbm_model, df, test_start, test_end, CLEANED_FEATURES)
    del lgbm_model
    
    garch_df = pd.read_parquet(str(garch_ckpt)) if garch_ckpt.exists() else None
    
    tft_df = pd.DataFrame(tft_metas); tft_df["tft_prob"] = tft_probs; tft_df["label"] = tft_labels
    lgbm_df = pd.DataFrame(lgbm_metas); lgbm_df["lgbm_prob"] = lgbm_probs
    merged = tft_df.merge(lgbm_df, on=["date","ticker"], how="inner")
    merged["window"] = i
    
    if garch_df is not None and "ticker" in garch_df.columns:
        merged = merged.merge(garch_df[["ticker","vol_5d","risk_flag"]], on="ticker", how="left")
        merged["vol_5d"] = merged["vol_5d"].fillna(40.0)
        merged["risk_flag"] = merged["risk_flag"].fillna(0).astype(int)
    else:
        merged["vol_5d"] = 40.0; merged["risk_flag"] = 0
    
    # v7: 파생 피처 생성 (메타모델용)
    merged["disagreement"] = (merged["tft_prob"] - merged["lgbm_prob"]).abs()
    merged["consensus_strength"] = merged["tft_prob"] * merged["lgbm_prob"]
    
    all_predictions.append(merged)
    print(f"       {len(merged):,} 예측")

pred_df = pd.concat(all_predictions, ignore_index=True)
print(f"\n전체 예측: {len(pred_df):,} rows, 윈도우 {pred_df['window'].nunique()}개")
print(f"메타 피처: {META_FEATURES}")

In [ ]:
# ===== Step 2: v2 슬라이딩 메타모델 (6피처 + 최근 4윈도우) =====
pred_df["ensemble_prob"] = np.nan

for i in range(BT_START_WINDOW, pred_df["window"].max() + 1):
    # 슬라이딩 윈도우: 최근 4윈도우만 학습
    train_mask = (pred_df["window"] >= max(0, i - SLIDING_WINDOW)) & (pred_df["window"] < i)
    test_mask = pred_df["window"] == i
    
    if train_mask.sum() < 100 or test_mask.sum() == 0:
        continue
    
    # 6피처 메타모델
    X_train = pred_df[train_mask][META_FEATURES].values
    y_train = pred_df[train_mask]["label"].values
    X_test = pred_df[test_mask][META_FEATURES].values
    
    meta = LogisticRegression(random_state=42, class_weight="balanced", max_iter=500)
    meta.fit(X_train, y_train)
    
    ensemble_probs = meta.predict_proba(X_test)[:, 1]
    pred_df.loc[test_mask, "ensemble_prob"] = ensemble_probs
    
    da = accuracy_score(pred_df[test_mask]["label"], (ensemble_probs >= 0.5).astype(int))
    coefs = {f: round(c, 2) for f, c in zip(META_FEATURES, meta.coef_[0])}
    print(f"  Window [{i:2d}] 학습={train_mask.sum():,}(W{max(0,i-SLIDING_WINDOW)}~{i-1}) → "
          f"DA={da*100:.1f}% | TFT={coefs['tft_prob']} LGBM={coefs['lgbm_prob']} "
          f"dis={coefs['disagreement']} con={coefs['consensus_strength']} "
          f"vol={coefs['vol_5d']} risk={coefs['risk_flag']}")

bt_df = pred_df.dropna(subset=["ensemble_prob"]).copy()
print(f"\n백테스트 대상: {len(bt_df):,} rows ({bt_df['date'].min()} ~ {bt_df['date'].max()})")

In [ ]:
# ===== Step 3: 날짜별 예측 dict 생성 =====
predictions = {}
for _, row in bt_df.iterrows():
    d, t = row["date"], row["ticker"]
    if d not in predictions: predictions[d] = {}
    predictions[d][t] = float(row["ensemble_prob"])

print(f"예측 거래일: {len(predictions)}, 기간: {min(predictions.keys())} ~ {max(predictions.keys())}")

In [ ]:
# ===== Step 4: 종가 데이터 로드 =====
tickers_needed = set()
for dp in predictions.values(): tickers_needed.update(dp.keys())

close_prices = {}
for ticker in tickers_needed:
    path = RAW_OHLCV_PATH / f"{ticker}.parquet"
    if not path.exists(): continue
    try:
        ohlcv = pd.read_parquet(str(path))
        ohlcv.index = pd.to_datetime(ohlcv.index)
        close_prices[ticker] = ohlcv["close"].sort_index()
    except: pass

kospi = pd.Series(dtype=float)
kospi_path = RAW_MACRO_PATH / "kospi200.parquet"
if kospi_path.exists():
    kdf = pd.read_parquet(str(kospi_path))
    kdf.index = pd.to_datetime(kdf.index)
    col = "close" if "close" in kdf.columns else kdf.columns[0]
    kospi = kdf[col].sort_index()

print(f"종가: {len(close_prices)} 종목, KOSPI200: {len(kospi)} rows")

In [ ]:
# ===== Step 5: 그리드 서치 — 여러 threshold로 시뮬레이션 반복 =====
@dataclass
class Position:
    ticker: str; buy_date: str; buy_price: float; shares: int
    buy_prob: float; invested: float; hold_days: int = 0
    extended: bool = False; peak_price: float = 0.0

def get_close(ticker, date):
    if ticker not in close_prices: return None
    s = close_prices[ticker]; ts = pd.Timestamp(date)
    if ts in s.index: return float(s.loc[ts])
    mask = s.index <= ts
    return float(s.loc[mask].iloc[-1]) if mask.any() else None

def run_backtest(predictions, buy_threshold, top_n, max_positions):
    """단일 파라미터 조합으로 백테스트 실행."""
    cash = INITIAL_CAPITAL
    positions = []; trade_log = []; snapshots = []
    cooldown = {}; skip_count = 0
    extend_stats = {"extended": 0, "timeout_base": 0, "timeout_max": 0}
    
    trading_dates = sorted(predictions.keys())
    
    for i, date in enumerate(trading_dates):
        for p in positions: p.hold_days += 1
        
        date_preds = predictions.get(date, {})
        sells = []
        for pos in positions:
            price = get_close(pos.ticker, date)
            if price is None: continue
            ret = (price - pos.buy_price) / pos.buy_price
            if pos.extended: pos.peak_price = max(pos.peak_price, price)
            
            sl = EXTENDED_STOP_LOSS if pos.extended else STOP_LOSS
            if ret <= sl:
                sells.append((pos, "stop_loss" if not pos.extended else "ext_stop_loss")); continue
            cp = date_preds.get(pos.ticker)
            if cp is not None and cp < MODEL_EXIT:
                sells.append((pos, "model_exit")); continue
            tp = EXTENDED_TAKE_PROFIT if pos.extended else TAKE_PROFIT
            if ret >= tp:
                sells.append((pos, "take_profit" if not pos.extended else "ext_take_profit")); continue
            if pos.hold_days >= MAX_HOLD_DAYS:
                extend_stats["timeout_max"] += 1
                sells.append((pos, "timeout_max")); continue
            if pos.hold_days >= BASE_HOLD_DAYS and not pos.extended:
                if cp is not None and cp >= HOLD_EXTEND_THRESHOLD and ret > 0:
                    pos.extended = True; pos.peak_price = price; extend_stats["extended"] += 1
                else:
                    extend_stats["timeout_base"] += 1
                    sells.append((pos, "timeout")); continue
            if pos.extended and pos.hold_days > BASE_HOLD_DAYS:
                if cp is not None and cp < HOLD_EXTEND_THRESHOLD:
                    sells.append((pos, "ext_signal_fade")); continue
        
        for pos, reason in sells:
            price = get_close(pos.ticker, date)
            if price is None: continue
            rev = pos.shares * price; comm = rev * SELL_COMMISSION; net = rev - comm
            cash += net; positions.remove(pos)
            pnl = net - pos.invested
            trade_log.append({"pnl": pnl, "return": pnl/pos.invested,
                "hold_days": pos.hold_days, "reason": reason, "extended": pos.extended})
            if "stop_loss" in reason:
                cooldown[pos.ticker] = i + REBUY_COOLDOWN
        
        preds = predictions[date]
        held = {p.ticker for p in positions}
        candidates = []
        for t, p in preds.items():
            if t in held: continue
            if p <= buy_threshold: continue
            if t in cooldown and i < cooldown[t]:
                skip_count += 1; continue
            candidates.append((t, p))
        candidates.sort(key=lambda x: -x[1])
        
        for ticker, prob in candidates[:top_n]:
            if len(positions) >= max_positions: break
            price = get_close(ticker, date)
            if price is None or price <= 0: continue
            slots = max_positions - len(positions)
            alloc = min(cash / max(slots, 1), cash)
            shares = int(alloc / (price * (1 + BUY_COMMISSION)))
            if shares <= 0: continue
            cost = shares * price; comm = cost * BUY_COMMISSION; total = cost + comm
            if total > cash: continue
            cash -= total
            positions.append(Position(ticker, date, price, shares, prob, total, peak_price=price))
        
        pv = cash + sum(p.shares * (get_close(p.ticker, date) or 0) for p in positions)
        snapshots.append({"date": date, "portfolio_value": pv})
    
    # 잔여 청산
    if positions:
        last = trading_dates[-1]
        for pos in list(positions):
            price = get_close(pos.ticker, last)
            if price is None: continue
            rev = pos.shares * price; comm = rev * SELL_COMMISSION; net = rev - comm
            cash += net; positions.remove(pos)
            trade_log.append({"pnl": net-pos.invested, "return": (net-pos.invested)/pos.invested,
                "hold_days": pos.hold_days, "reason": "backtest_end", "extended": pos.extended})
    
    # 결과 계산
    snap_df = pd.DataFrame(snapshots)
    snap_df["date"] = pd.to_datetime(snap_df["date"])
    sell_df = pd.DataFrame(trade_log)
    
    final = snap_df["portfolio_value"].iloc[-1]
    total_ret = final / INITIAL_CAPITAL - 1
    daily_ret = snap_df["portfolio_value"].pct_change().dropna()
    sharpe = float(daily_ret.mean() / daily_ret.std() * np.sqrt(252)) if daily_ret.std() > 1e-9 else 0
    mdd = float(((snap_df["portfolio_value"] - snap_df["portfolio_value"].cummax()) / snap_df["portfolio_value"].cummax()).min())
    
    win_rate = (sell_df["pnl"] > 0).mean() if len(sell_df) > 0 else 0
    n_trades = len(sell_df)
    ext_sells = sell_df[sell_df["extended"] == True] if len(sell_df) > 0 else pd.DataFrame()
    ext_ratio = len(ext_sells) / max(n_trades, 1) * 100
    ext_wr = (ext_sells["pnl"] > 0).mean() * 100 if len(ext_sells) > 0 else 0
    
    return {
        "total_return": round(total_ret * 100, 2),
        "sharpe": round(sharpe, 3),
        "mdd": round(mdd * 100, 2),
        "n_trades": n_trades,
        "win_rate": round(win_rate * 100, 1),
        "ext_ratio": round(ext_ratio, 1),
        "ext_wr": round(ext_wr, 1),
        "cooldown_skips": skip_count,
    }

# 그리드 서치 실행
print(f"그리드 서치 시작 ({len(GRID)}개 조합)\n")
grid_results = []

for g in GRID:
    bt = g["buy_threshold"]
    tn = g["top_n"]
    mp = g["max_positions"]
    
    result = run_backtest(predictions, bt, tn, mp)
    result.update({"buy_threshold": bt, "top_n": tn, "max_positions": mp})
    grid_results.append(result)
    
    print(f"  BT={bt} TN={tn} MP={mp} → "
          f"수익={result['total_return']:+.1f}% Sharpe={result['sharpe']:.3f} "
          f"MDD={result['mdd']:.1f}% 거래={result['n_trades']} "
          f"승률={result['win_rate']:.1f}% 연장={result['ext_ratio']:.0f}%")

print(f"\n그리드 서치 완료")

In [ ]:
# ===== Step 6: 그리드 서치 결과 비교 =====
print("="*90)
print("  v7b 그리드 서치 결과 (v2 슬라이딩 메타모델 + BUY_THRESHOLD 탐색)")
print("="*90)

print(f"\n  {'BT':>5} {'TN':>4} {'MP':>4} {'수익률':>8} {'Sharpe':>8} {'MDD':>8} {'거래':>6} {'승률':>6} {'연장%':>6} {'연장WR':>7}")
print(f"  {'-'*72}")

best = max(grid_results, key=lambda x: x["sharpe"])

for r in grid_results:
    marker = " ★" if r == best else ""
    print(f"  {r['buy_threshold']:>5.2f} {r['top_n']:>4} {r['max_positions']:>4} "
          f"{r['total_return']:>+7.1f}% {r['sharpe']:>7.3f} {r['mdd']:>7.1f}% "
          f"{r['n_trades']:>5} {r['win_rate']:>5.1f}% {r['ext_ratio']:>5.1f}% "
          f"{r['ext_wr']:>6.1f}%{marker}")

print(f"\n  ★ 최적 (Sharpe 기준): BT={best['buy_threshold']}, "
      f"수익={best['total_return']:+.1f}%, Sharpe={best['sharpe']:.3f}, MDD={best['mdd']:.1f}%")

# v6 대비
print(f"\n  v6 참고: 수익=+89.02%, Sharpe=0.985, MDD=-14.82% (v1 메타모델)")
print("="*90)

In [ ]:
# ===== 저장 =====
json.dump(grid_results, open(str(MODEL_SAVE_PATH / "grid_search_v7b.json"), "w"), indent=2)
print(f"저장: {MODEL_SAVE_PATH / 'grid_search_v7b.json'}")

## v7b: BUY_THRESHOLD 그리드 서치

### 구조
```
Step 1~4: v7과 동일 (v2 슬라이딩 메타모델로 ensemble_prob 생성)
Step 5: 7개 threshold 조합으로 백테스트 반복
Step 6: 결과 비교표
```

### 탐색 범위
| BT | TOP_N | MAX_POS | 설계 의도 |
|----|-------|---------|----------|
| 0.50 | 10 | 20 | v2 원본 수준 (넓은 필터) |
| 0.52 | 8 | 18 | 약간 필터 |
| 0.54 | 7 | 15 | v7 원본 |
| 0.56 | 7 | 15 | 조금 높임 |
| 0.58 | 7 | 15 | v3 수준 |
| 0.60 | 5 | 12 | 강한 필터 |
| 0.62 | 5 | 10 | 매우 강한 필터 |